In [18]:
!pip install openai wikipedia python-docx rich
!pip install langchain langchain-community streamlit


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import wikipedia
from docx import Document


class WikiSearchTool:

    def search(self, query):

        try:
            result = wikipedia.summary(query, sentences=5)
            return result

        except Exception as e:
            return f"Error: {e}"


class DocumentTool:

    def create_document(self, title, content):

        doc = Document()

        doc.add_heading(title, level=1)
        doc.add_paragraph(content)

        filename = f"documents/{title}.docx"

        doc.save(filename)

        return filename

In [21]:
import wikipedia
from docx import Document


class WikiSearchTool:

    def search(self, query):

        try:
            result = wikipedia.summary(query, sentences=5)
            return result

        except Exception as e:
            return f"Error: {e}"


class DocumentTool:

    def create_document(self, title, content):

        doc = Document()

        doc.add_heading(title, level=1)
        doc.add_paragraph(content)

        filename = f"{title}.docx"

        doc.save(filename)

        return filename

In [47]:
import sys
import re
from openai import OpenAI
import wikipedia
from docx import Document


# =====================================
# TOOLS
# =====================================

class WikiSearchTool:

    def search(self, query):

        try:
            result = wikipedia.summary(query, sentences=5)
            return result

        except Exception:
            return "Could not retrieve information from Wikipedia."


class DocumentTool:

    def create_document(self, title, content):

        try:

            doc = Document()

            doc.add_heading(title, level=1)
            doc.add_paragraph(content)

            filename = f"{title}.docx"

            doc.save(filename)

            return filename

        except Exception as e:

            return f"Document Error: {e}"


# =====================================
# AGENTS
# =====================================

class WikiAgent:

    def __init__(self):
        self.tool = WikiSearchTool()

    def run(self, topic):
        return self.tool.search(topic)


class DocumentAgent:

    def __init__(self):
        self.tool = DocumentTool()

    def run(self, title, content):
        return self.tool.create_document(title, content)


# =====================================
# LM STUDIO CONNECTION
# =====================================

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)


wiki_agent = WikiAgent()
doc_agent = DocumentAgent()


# =====================================
# SYSTEM PROMPT
# =====================================

SYSTEM_PROMPT = """
You are a helpful AI assistant.

Rules:

1. If the user asks to search, define, explain, or look up a topic,
reply ONLY in this format:

WIKI: topic

2. If the user asks to create, save, export, or generate a document,
reply ONLY in this format:

DOC: title | content

3. Otherwise answer normally.
"""


messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]


# =====================================
# MEMORY + STATS
# =====================================

conversation_history = []

conversation_stats = {
    "user_turns": 0,
    "wiki_searches": 0,
    "document_saves": 0,
    "ai_chats": 0
}


# =====================================
# STARTUP MESSAGE
# =====================================

print("Agentic AI Chatbot initialized!")
print("Type 'exit' to quit.\n")


# =====================================
# CHAT LOOP
# =====================================

while True:

    user_input = input("You: ")

    # EXIT
    if user_input.lower() == "exit":

        print("\nConversation Statistics:\n")

        for key, value in conversation_stats.items():
            print(f"{key}: {value}")

        print("\nGoodbye!")
        break

    # EMPTY INPUT
    if not user_input.strip():
        continue

    conversation_stats["user_turns"] += 1

    messages.append({
        "role": "user",
        "content": user_input
    })

    try:

        # =====================================
        # TOOL DECISION
        # =====================================

        response = client.chat.completions.create(
            model="meta-llama-3.1-8b-instruct",
            messages=messages,
            temperature=0.3
        )

        decision = response.choices[0].message.content.strip()

        # =====================================
        # WIKIPEDIA SEARCH
        # =====================================

        if decision.upper().startswith("WIKI:"):

            conversation_stats["wiki_searches"] += 1

            topic = decision.replace("WIKI:", "").strip()

            wiki_result = wiki_agent.run(topic)

            print(f"\nUser: {user_input}\n")

            print("Assistant:\n")
            print(f"WIKI: {topic}\n")
            print(wiki_result)
            print()

            conversation_history.append(
                (user_input, wiki_result)
            )

            messages.append({
                "role": "assistant",
                "content": wiki_result
            })

            continue

        # =====================================
        # DOCUMENT CREATION
        # =====================================

        elif decision.upper().startswith("DOC:"):

            conversation_stats["document_saves"] += 1

            try:

                raw_data = decision.replace("DOC:", "").strip()

                title, content = raw_data.split("|", 1)

                title = title.strip()
                content = content.strip()

            except ValueError:

                title = "Generated_Document"
                content = decision

            filename = doc_agent.run(title, content)

            assistant_reply = f"Document saved successfully as '{filename}'"

            print(f"\nUser: {user_input}\n")

            print("Assistant:\n")
            print(assistant_reply)
            print()

            conversation_history.append(
                (user_input, assistant_reply)
            )

            messages.append({
                "role": "assistant",
                "content": assistant_reply
            })

            continue

        # =====================================
        # NORMAL CHAT
        # =====================================

        else:

            conversation_stats["ai_chats"] += 1

            print(f"\nUser: {user_input}\n")

            print("Assistant:\n")

            stream = client.chat.completions.create(
                model="meta-llama-3.1-8b-instruct",
                messages=messages,
                temperature=0.7,
                stream=True
            )

            assistant_reply = ""

            for chunk in stream:

                content = chunk.choices[0].delta.content

                if content:

                    sys.stdout.write(content)
                    sys.stdout.flush()

                    assistant_reply += content

            print("\n")

            conversation_history.append(
                (user_input, assistant_reply)
            )

            messages.append({
                "role": "assistant",
                "content": assistant_reply
            })

    except Exception as e:

        print(f"\nError: {e}\n")

Agentic AI Chatbot initialized!

Type 'exit' to quit.

User: Hello, I'm Gwen a 3rd yr Computer Engineering Student

Assistant:

Nice to meet you, Gwen! How's life as a 3rd-year Computer Engineering student treating you? Are you enjoying the courses and projects so far? Do you have any specific areas of interest or challenges in your studies that you'd like to discuss?

...

user_turns: 7

wiki_searches: 1

document_saves: 1

ai_chats: 5

Goodbye!